## 00 - Audio 2 Vec - Starter code
In this notebook, we will convert audio files directly to embeddings, skipping the text transription step

In [1]:
# !pip install flash-attn

In [2]:
# !pip install transformers

In [3]:
# !pip install torch

In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.current_device())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

True
NVIDIA GeForce RTX 3060 Laptop GPU
0
cuda


In [5]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor


In [6]:
model_name = "facebook/wav2vec2-large-960h-lv60-self"
model = Wav2Vec2Model.from_pretrained(model_name,
                                     torch_dtype=torch.float16,
                                     ).to(device)
                                     # )

C:\Users\Minh UBC\miniconda3\envs\dsci512\Lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name,
                                                    torch_dtype=torch.float16)

In [8]:
# !pip install librosa

In [9]:
import librosa
# load wav file
# fname = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/cn/adrso002.wav"
fname = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/adrso024.wav"
print("Loading file =", fname)
input_audio, sample_rate = librosa.load(fname, sr=16000)
print("Completed")

Loading file = ../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/adrso024.wav
Completed


In [10]:
# convert to float16
import numpy as np

input_audio = input_audio.astype(np.float16)
result = extractor(input_audio, return_tensors="pt", sampling_rate=sample_rate).to(device)


In [11]:
print(type(result))
# print(result.shape)

<class 'transformers.feature_extraction_utils.BatchFeature'>


In [12]:
with torch.no_grad():
    output = model(result.input_values)

print(output.keys())
print(output.last_hidden_state.shape)
print(output.extract_features.shape)

odict_keys(['last_hidden_state', 'extract_features'])
torch.Size([1, 11792, 1024])
torch.Size([1, 11792, 512])


In [13]:
# vecs = output.extract_features.cpu()  # copy to cpu to use np
vecs = output.extract_features
print(vecs.shape)

torch.Size([1, 11792, 512])


In [14]:
# calculate mean pooling
# import numpy as np
# v = np.average(vecs, axis=1)
# print(v.shape)

In [15]:
# get size of vecs
import sys
sys.getsizeof(vecs)

88

In [16]:
v = torch.mean(vecs, dim=1)

In [17]:
print(v.shape)


torch.Size([1, 512])


In [18]:
print(vecs[0, :, 0])
print(torch.mean(vecs[0], dim=0)[0])

tensor([ 0.7192,  0.3276, -0.3816,  ..., -0.2698, -0.0908,  0.2397],
       device='cuda:0', dtype=torch.float16)
tensor(0.0743, device='cuda:0', dtype=torch.float16)


In [19]:
print(v[0][0])

tensor(0.0743, device='cuda:0', dtype=torch.float16)


In [20]:
# put into a function
def audio2vec(fname, model=model, extractor=extractor, fl16=True):
    """
        Read a wav file and return mean pooling embeddings (length 512)
    """
    result = []
    print("Loading file =", fname)
    input_audio, sample_rate = librosa.load(fname, sr=16000)

    # convert to float16
    if fl16:
        input_audio = input_audio.astype(np.float16)
        
    print("Completed")
    print("Calculating embeddings...")
    
    result = extractor(input_audio, return_tensors="pt", sampling_rate=sample_rate).to(device)
    with torch.no_grad():
        output = model(result.input_values)

    print("Calculating mean pooling...")

    # result = np.average(output.extract_features[0], axis=0)
    result = torch.mean(output.extract_features[0], dim=0)
    # debug:
    print("Shape =", result.shape)
    return result

In [21]:
# fname = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/cn/adrso002.wav"  # small file 6 MB
# fname = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/adrso024.wav" # bigger file 20 MB
# fname = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/adrso192.wav" # bigger file 38 MB
fname = "../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/adrso056.wav" # bigger file 90 MB

result = audio2vec(fname)
print(result[:10])

Loading file = ../../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/adrso056.wav
Completed
Calculating embeddings...
Calculating mean pooling...
Shape = torch.Size([512])
tensor([ 0.2024, -0.3721, -0.2156,  0.2493,  0.5205,  0.0225, -0.5479,  0.6021,
        -0.6152, -0.2471], device='cuda:0', dtype=torch.float16)


### test transribe using speechbrain

In [22]:
# !pip install speechbrain

In [23]:
# from speechbrain.inference import EncoderDecoderASR
# import speechbrain as sb